# Task 2 — NER (`LOCATION`, `ORGANIZATION`, `PERSON`) & LDA Topic Modeling (Google Colab Runner)

Notebook ini menjalankan seluruh pipeline **Task 2 (T2)** sesuai rancangan:
- **Phase 1 & 2 · E1 CPU Baseline:** Feature-based BIO2 + Cased Civic Gazetteer & 3-Gate `, ` Post-Processing
- **Phase 3 · LDA Topic Modeling:** 4-Layer Stopwords + Gensim `Phrases` Bigrams + Sweep `K in {5, 8, 10, 12, 15}` (fit ketat pada `split == "train"`)
- **Phase 4 · Transformer NER (E2–E4):** Fine-tuning **IndoBERT** (`indobenchmark/indobert-base-p1`, `max_length=384`, dynamic padding, `--fp16` + `GradScaler` guard) untuk:
  - `ner_full` (Real + Synthetic, 3 seed: `42, 123, 2025`) $\rightarrow$ **E2** (Raw) & **E3** (Main: + Civic Post-Processing)
  - `ner_real_only` (Real-Only Ablation, 3 seed: `42, 123, 2025`) $\rightarrow$ **E4a** (Raw) & **E4b** (+ Civic Post-Processing)
- **Phase 5 · Verifikasi & Packaging:** Unit test + inferensi `T2Predictor` + arsip scoped `outputs/task2_*.json` & `models/task2/`

In [ ]:
!nvidia-smi

In [ ]:
import os
from getpass import getpass

PROJECT_DIR = "/content/Tubes_NLP"
if not os.path.exists(PROJECT_DIR):
    token = getpass("Masukkan GitHub Token: ")
    REPO_URL = f"https://{token}@github.com/RylRizqi16/Tubes_NLP.git"
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

master_path = Path("data/processed/master_dataset.jsonl")
if not master_path.exists():
    print("master_dataset.jsonl belum ada, mengunduh & membangun dataset...")
    !python scripts/download_datasets.py
    !python scripts/generate_synthetic_edge_cases.py
    !python scripts/build_master_dataset.py
else:
    print(f"Dataset siap: {master_path} ({master_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
!python -m src.task2.train_ner_baseline

In [ ]:
!python -m src.task2.train_lda --backend gensim --k-values 5 8 10 12 15 --seed 42

In [ ]:
!python -m src.task2.train_ner_transformer \
    --model-name indobenchmark/indobert-base-p1 \
    --seeds 42 123 2025 \
    --epochs 5 \
    --batch-size 16 \
    --lr 3e-5 \
    --max-length 384 \
    --patience 2 \
    --fp16

In [ ]:
!python -m src.task2.train_ner_transformer \
    --model-name indobenchmark/indobert-base-p1 \
    --seeds 42 123 2025 \
    --epochs 5 \
    --batch-size 16 \
    --lr 3e-5 \
    --max-length 384 \
    --patience 2 \
    --fp16 \
    --real-only

In [ ]:
!python -m src.task2.evaluate_ner --split test

In [ ]:
!python -m unittest src.task2.test_task2 -v

from src.task2.predictor import T2Predictor

t2 = T2Predictor()
print(f"Active NER backend: {t2.ner_backend} (checkpoint={t2.checkpoint_dir})")
samples = [
    "Lampu jalan di Jl. Setiabudi, Srondol mati 5 hari, mohon Dishub Kota Semarang segera perbaiki.",
    "Terima kasih Pak Budi dan Puskesmas Tembalang atas pelayanan cepat hari ini.",
    "Aplikasi SP4N LAPOR sering error saat upload foto laporan warga.",
]
for text in samples:
    print("\nTEXT:", text)
    print("  NER  :", t2.predict(text))
    topic = t2.predict_topic(text)
    print(f"  TOPIC: topic_{topic['topic_id']} (p={topic['topic_probability']:.4f}) -> {topic['top_keywords'][:6]}")

In [ ]:
!zip -rq task2_colab_results.zip outputs/task2_*.json models/task2/
print("Siap diunduh: task2_colab_results.zip")